In [1]:
# 03_sqlite – 02'nin ara çıktılarını yükle
from pathlib import Path
import pandas as pd

ISLENMIS = Path("../data/processed")

oyunlar       = pd.read_parquet(ISLENMIS / "oyunlar.parquet")
oyun_tur      = pd.read_parquet(ISLENMIS / "oyun_tur.parquet")
oyun_etiket   = pd.read_parquet(ISLENMIS / "oyun_etiket.parquet")
oyun_kategori = pd.read_parquet(ISLENMIS / "oyun_kategori.parquet")
oyun_dil      = pd.read_parquet(ISLENMIS / "oyun_dil.parquet")

yan_tablolar = {"oyun_tur": oyun_tur, "oyun_etiket": oyun_etiket,
                "oyun_kategori": oyun_kategori, "oyun_dil": oyun_dil}

print(f"oyunlar: {oyunlar.shape[0]:,} satır × {oyunlar.shape[1]} sütun (index: {oyunlar.index.name})")
for ad, tablo in yan_tablolar.items():
    yetim = (~tablo["app_id"].isin(oyunlar.index)).sum()
    print(f"{ad}: {tablo.shape[0]:,} satır, sütunlar {list(tablo.columns)}, oyunlar'da olmayan app_id: {yetim}")

oyunlar.info()

oyunlar: 64,948 satır × 35 sütun (index: app_id)
oyun_tur: 174,563 satır, sütunlar ['app_id', 'tur'], oyunlar'da olmayan app_id: 0
oyun_etiket: 730,709 satır, sütunlar ['app_id', 'etiket', 'oy', 'sira'], oyunlar'da olmayan app_id: 0
oyun_kategori: 235,168 satır, sütunlar ['app_id', 'kategori'], oyunlar'da olmayan app_id: 0
oyun_dil: 341,659 satır, sütunlar ['app_id', 'dil', 'ses_var'], oyunlar'da olmayan app_id: 0
<class 'pandas.DataFrame'>
Index: 64948 entries, 1724540 to 1504030
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   ad                        64948 non-null  str           
 1   steam_cikis_tarihi        64948 non-null  datetime64[us]
 2   fiyat                     64948 non-null  float64       
 3   indirim_yuzde             51556 non-null  Int64         
 4   normal_fiyat              64948 non-null  float64       
 5   yas_siniri                422 non-null    

## 1. 02'nin ara çıktılarını yükleme

`data/processed/` altındaki 5 parquet dosyası okundu:

| Tablo | Satır | Sütunlar |
|---|---|---|
| oyunlar | 64.948 | 35 sütun, index app_id |
| oyun_tur | 174.563 | app_id, tur |
| oyun_etiket | 730.709 | app_id, etiket, oy, sira |
| oyun_kategori | 235.168 | app_id, kategori |
| oyun_dil | 341.659 | app_id, dil, ses_var |

**Kontroller:**
- Yan tablolarda oyunlar'da karşılığı olmayan app_id sayısı: **0** (dördünde de). SQLite'ta app_id ile bağlanınca boşta satır kalmayacak.
- Yorumdan gelen sütunlar (ilk/son_yorum_tarihi, yorumcu_medyan_saat) **22.898** oyunda dolu = 23.107 CSV − 209 yazılım dosyası.

**SQLite'a geçerken dikkat:** pandas tipleri Int64, float64, str, bool ve datetime64. SQLite'ta ayrı bir tarih ve bool tipi yok; bunların nasıl saklanacağı şema adımında kararlaştırılacak.

In [2]:
# Tip testi: pandas tipleri SQLite'a nasıl geçiyor?
import sqlite3

ornek = pd.concat([
    oyunlar[oyunlar["ilk_yorum_tarihi"].notna()].head(2),
    oyunlar[oyunlar["ilk_yorum_tarihi"].isna()].head(1),
])
test_sutunlar = ["ad", "steam_cikis_tarihi", "ilk_yorum_tarihi", "yetiskin",
                 "yorum_var", "olumlu_oran", "yas_siniri"]

test_db = sqlite3.connect(":memory:")
ornek[test_sutunlar].to_sql("oyunlar_test", test_db, index=True)

print(test_db.execute("SELECT sql FROM sqlite_master WHERE name = 'oyunlar_test'").fetchone()[0])
print()
for satir in test_db.execute("SELECT * FROM oyunlar_test"):
    print(satir)
print()
print(test_db.execute("""SELECT typeof(steam_cikis_tarihi), typeof(ilk_yorum_tarihi),
                                typeof(yetiskin), typeof(yorum_var), typeof(yas_siniri)
                         FROM oyunlar_test""").fetchall())
test_db.close()

CREATE TABLE "oyunlar_test" (
"app_id" TEXT,
  "ad" TEXT,
  "steam_cikis_tarihi" TIMESTAMP,
  "ilk_yorum_tarihi" TIMESTAMP,
  "yetiskin" INTEGER,
  "yorum_var" INTEGER,
  "olumlu_oran" REAL,
  "yas_siniri" INTEGER
)

('1724540', 'Welcome to Goodland', '2023-04-28 00:00:00', '2023-04-28 00:00:00', 0, 1, 0.758, None)
('2218750', 'Halls of Torment', '2024-09-24 00:00:00', '2023-05-24 00:00:00', 0, 1, 0.958, None)
('2201840', '그녀가 다시 노래할 때까지.', '2022-11-14 00:00:00', None, 0, 0, 0.8, None)

[('text', 'text', 'integer', 'integer', 'null'), ('text', 'text', 'integer', 'integer', 'null'), ('text', 'null', 'integer', 'integer', 'null')]


## 2. Tip testi: pandas → SQLite

3 satırlık bellek içi denemede (`sqlite3.connect(":memory:")`) görülenler:

| pandas tipi | SQLite'ta | Karar |
|---|---|---|
| app_id (index, str) | TEXT `'1724540'` | **Tam sayıya (int64) çevrilecek** |
| datetime64 | TEXT `'2023-04-28 00:00:00'` | **Saatsiz `'2023-04-28'` metin** |
| bool / boolean | INTEGER 1/0 | Aynen kalır |
| boş değer (NaT, NA) | NULL | Aynen kalır |

**Neden:**
- app_id metin kalırsa Chroma metadata filtreleri ve Python tarafında tip uyuşmazlığı sessizce boş sonuç verir; `INTEGER PRIMARY KEY` olunca SQLite satıra doğrudan gider.
- Tarihte saat kalırsa `WHERE tarih = '2023-05-24'` hiçbir satır döndürmez. `YYYY-AA-GG` metin doğru sıralanır ve `strftime` ile çalışır.

**Örnek:** Halls of Torment: steam_cikis_tarihi 2024-09-24, ilk_yorum_tarihi 2023-05-24 (erken erişim). Oyunun yaşı için ilk_yorum_tarihi kullanılır.

In [3]:
# SQLite'a yazma 1: tip düzeltme + oyunlar tablosu
DB_YOLU = ISLENMIS / "steam.db"

def sqlite_icin_hazirla(df):
    df = df.copy()
    if df.index.name == "app_id":
        df = df.reset_index()
    df["app_id"] = df["app_id"].astype("int64")
    for sutun in df.select_dtypes(include="datetime").columns:
        df[sutun] = df[sutun].dt.strftime("%Y-%m-%d")
    return df

oyunlar_sql = sqlite_icin_hazirla(oyunlar)

DB_YOLU.unlink(missing_ok=True)
baglanti = sqlite3.connect(DB_YOLU)

sema = pd.io.sql.get_schema(oyunlar_sql, "oyunlar", keys="app_id", con=baglanti)
print(sema)
baglanti.execute(sema)
oyunlar_sql.to_sql("oyunlar", baglanti, if_exists="append", index=False)
baglanti.commit()

print()
print("Satır sayısı:", baglanti.execute("SELECT COUNT(*) FROM oyunlar").fetchone()[0])
print(baglanti.execute("""SELECT app_id, typeof(app_id), ad, steam_cikis_tarihi, ilk_yorum_tarihi
                          FROM oyunlar WHERE app_id = 2218750""").fetchone())
print("ilk_yorum_tarihi = '2023-05-24' olan oyun sayısı:",
      baglanti.execute("SELECT COUNT(*) FROM oyunlar WHERE ilk_yorum_tarihi = '2023-05-24'").fetchone()[0])

CREATE TABLE "oyunlar" (
"app_id" INTEGER,
  "ad" TEXT,
  "steam_cikis_tarihi" TEXT,
  "fiyat" REAL,
  "indirim_yuzde" INTEGER,
  "normal_fiyat" REAL,
  "yas_siniri" INTEGER,
  "sahip_alt" INTEGER,
  "sahip_ust" INTEGER,
  "ortalama_sure_saat" REAL,
  "medyan_sure_saat" REAL,
  "peak_ccu" INTEGER,
  "istatistik_eksik" INTEGER,
  "olumlu_json" INTEGER,
  "olumsuz_json" INTEGER,
  "olumlu_csv" INTEGER,
  "olumsuz_csv" INTEGER,
  "oran_kaynagi" TEXT,
  "oy_sayisi" INTEGER,
  "olumlu_oran" REAL,
  "yorum_sayisi" INTEGER,
  "yorum_var" INTEGER,
  "yorum_kapsami" REAL,
  "kisa_aciklama" TEXT,
  "oyun_hakkinda" TEXT,
  "iskelet" INTEGER,
  "yetiskin" INTEGER,
  "yetiskin_etiket_sayisi" INTEGER,
  "dil_spam" INTEGER,
  "ses_spam" INTEGER,
  "ilk_yorum_tarihi" TEXT,
  "son_yorum_tarihi" TEXT,
  "yorumcu_medyan_saat" REAL,
  "yorumcu_sure_sifir_orani" REAL,
  "tr_yorum_sayisi" INTEGER,
  "chroma_yorum_sayisi" INTEGER,
  CONSTRAINT oyunlar_pk PRIMARY KEY ("app_id")
)

Satır sayısı: 64948
(2218750

## 3. oyunlar tablosu → steam.db

Veritabanı: `data/processed/steam.db` (tek dosya, gitignore'da).

**Yöntem:**
- `sqlite_icin_hazirla(df)`: app_id'yi index'ten sütuna alır ve int64'e çevirir; tarih sütunlarını `'YYYY-AA-GG'` metnine çevirir (boş tarih → NULL).
- `pd.io.sql.get_schema(..., keys="app_id")` ile birincil anahtarlı `CREATE TABLE` üretildi, çünkü `to_sql` tek başına anahtar koyamıyor. Tablo bu şemayla kuruldu, veri `to_sql(if_exists="append")` ile eklendi.

**Sonuç:**
- 64.948 satır, 36 sütun (app_id + 35).
- app_id `INTEGER PRIMARY KEY` → SQLite'ın rowid'i; app_id ile arama doğrudan satıra gider.
- Kontrol: Halls of Torment (2218750) → `typeof(app_id) = integer`, steam_cikis_tarihi `2024-09-24`, ilk_yorum_tarihi `2023-05-24`.
- `WHERE ilk_yorum_tarihi = '2023-05-24'` → 17 oyun (saatli biçimde 0 dönerdi).

Not: Bu hücre her çalıştığında `steam.db` silinip baştan kurulur.

In [4]:
# SQLite'a yazma 2: yan tablolar (birleşik anahtar + oyunlar'a bağlantı)
baglanti.execute("PRAGMA foreign_keys = ON")

yan_semalar = {
    "oyun_tur": ("""CREATE TABLE oyun_tur (
        app_id INTEGER NOT NULL REFERENCES oyunlar(app_id),
        tur    TEXT    NOT NULL,
        PRIMARY KEY (app_id, tur))""", ["app_id", "tur"]),
    "oyun_etiket": ("""CREATE TABLE oyun_etiket (
        app_id INTEGER NOT NULL REFERENCES oyunlar(app_id),
        etiket TEXT    NOT NULL,
        oy     INTEGER,
        sira   INTEGER,
        PRIMARY KEY (app_id, etiket))""", ["app_id", "etiket"]),
    "oyun_kategori": ("""CREATE TABLE oyun_kategori (
        app_id   INTEGER NOT NULL REFERENCES oyunlar(app_id),
        kategori TEXT    NOT NULL,
        PRIMARY KEY (app_id, kategori))""", ["app_id", "kategori"]),
    "oyun_dil": ("""CREATE TABLE oyun_dil (
        app_id  INTEGER NOT NULL REFERENCES oyunlar(app_id),
        dil     TEXT    NOT NULL,
        ses_var INTEGER,
        PRIMARY KEY (app_id, dil))""", ["app_id", "dil"]),
}

for ad, (sema, anahtar) in yan_semalar.items():
    tablo_sql = sqlite_icin_hazirla(yan_tablolar[ad])
    tekrar = tablo_sql.duplicated(subset=anahtar).sum()
    print(f"{ad}: {len(tablo_sql):,} satır, anahtar tekrarı: {tekrar}")
    if tekrar > 0:
        print(f"  → {ad} YAZILMADI, tekrarlara bakmamız lazım")
        continue
    baglanti.execute(f"DROP TABLE IF EXISTS {ad}")
    baglanti.execute(sema)
    tablo_sql.to_sql(ad, baglanti, if_exists="append", index=False)
baglanti.commit()

print()
for ad in yan_semalar:
    print(f"{ad} (SQLite): {baglanti.execute(f'SELECT COUNT(*) FROM {ad}').fetchone()[0]:,} satır")
print("foreign_key_check:", baglanti.execute("PRAGMA foreign_key_check").fetchall())
print()
for satir in baglanti.execute("""
        SELECT o.ad, e.sira, e.etiket, e.oy
        FROM oyun_etiket e
        JOIN oyunlar o ON o.app_id = e.app_id
        WHERE o.app_id = 2218750
        ORDER BY e.sira"""):
    print(satir)

oyun_tur: 174,563 satır, anahtar tekrarı: 0
oyun_etiket: 730,709 satır, anahtar tekrarı: 0
oyun_kategori: 235,168 satır, anahtar tekrarı: 0
oyun_dil: 341,659 satır, anahtar tekrarı: 0

oyun_tur (SQLite): 174,563 satır
oyun_etiket (SQLite): 730,709 satır
oyun_kategori (SQLite): 235,168 satır
oyun_dil (SQLite): 341,659 satır
foreign_key_check: []

('Halls of Torment', 1, 'Action Roguelike', 662)
('Halls of Torment', 2, 'Rogue-lite', 656)
('Halls of Torment', 3, 'Bullet Hell', 624)
('Halls of Torment', 4, 'Pixel Graphics', 596)
('Halls of Torment', 5, 'Action RPG', 590)
('Halls of Torment', 6, 'Dark Fantasy', 586)
('Halls of Torment', 7, 'Rogue-like', 583)
('Halls of Torment', 8, 'Action', 563)
('Halls of Torment', 9, 'RPG', 541)
('Halls of Torment', 10, 'Arcade', 505)
('Halls of Torment', 11, 'Hack and Slash', 466)
('Halls of Torment', 12, 'Top-Down', 458)
('Halls of Torment', 13, '2D', 455)
('Halls of Torment', 14, 'Singleplayer', 449)
('Halls of Torment', 15, 'Atmospheric', 448)
('Hall

## 4. Yan tablolar → steam.db

Tablolar `CREATE TABLE` ile elle kuruldu, çünkü `get_schema` yabancı anahtar ekleyemiyor. `PRAGMA foreign_keys = ON` ile kural denetimi açıldı (SQLite'ta varsayılan kapalı).

| Tablo | Satır | Birincil anahtar | Diğer sütunlar |
|---|---|---|---|
| oyun_tur | 174.563 | (app_id, tur) | |
| oyun_etiket | 730.709 | (app_id, etiket) | oy, sira |
| oyun_kategori | 235.168 | (app_id, kategori) | |
| oyun_dil | 341.659 | (app_id, dil) | ses_var |

Hepsinde `app_id REFERENCES oyunlar(app_id)` (yabancı anahtar).

**Kontroller:**
- Anahtar tekrarı: 4 tabloda da 0.
- SQLite satır sayıları parquet ile birebir aynı.
- `PRAGMA foreign_key_check` → `[]` (kuralı bozan satır yok).
- İlk JOIN: Halls of Torment'in 20 etiketi oy sırasıyla (1. Action Roguelike 662 oy … 20. Medieval 434 oy). "İlk N etiket" sorusu `WHERE sira <= N` ile cevaplanır.

**Durum:** 6 tablonun 5'i yazıldı. Kalan: yorumlar (~31,8 M satır, metin dahil).

In [5]:
# Yorumlar 1: dosya listesi + tek dosyada yapı kontrolü
YORUM_KLASORU = ISLENMIS / "yorumlar"

yorum_dosyalari = sorted(YORUM_KLASORU.glob("*.parquet"))
oyun_idleri = set(oyunlar.index.astype("int64"))
yazilacak = [d for d in yorum_dosyalari if int(d.stem) in oyun_idleri]

print(f"Klasördeki dosya: {len(yorum_dosyalari):,}")
print(f"oyunlar'a ait (yazılacak): {len(yazilacak):,}")
print(f"Atlanacak (yazılım): {len(yorum_dosyalari) - len(yazilacak):,}")
print(f"Beklenen satır (oyunlar.yorum_sayisi toplamı): {oyunlar['yorum_sayisi'].sum():,}")

ornek_yorum = pd.read_parquet(YORUM_KLASORU / "2218750.parquet")
print(f"\nHalls of Torment: {len(ornek_yorum):,} yorum × {ornek_yorum.shape[1]} sütun\n")
ornek_yorum.info()
print()
print(ornek_yorum.head(3).to_string())

Klasördeki dosya: 23,107
oyunlar'a ait (yazılacak): 22,898
Atlanacak (yazılım): 209
Beklenen satır (oyunlar.yorum_sayisi toplamı): 31,690,738

Halls of Torment: 10,820 yorum × 15 sütun

<class 'pandas.DataFrame'>
RangeIndex: 10820 entries, 0 to 10819
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   review_id     10820 non-null  str           
 1   app_id        10820 non-null  str           
 2   oynama_saat   10820 non-null  float64       
 3   tarih         10820 non-null  datetime64[us]
 4   faydali_oy    10820 non-null  int64         
 5   metin         10760 non-null  str           
 6   olumlu        10820 non-null  bool          
 7   erken_erisim  10820 non-null  bool          
 8   uzunluk       10820 non-null  int64         
 9   dil_ham       10760 non-null  str           
 10  dil_guven     10820 non-null  float64       
 11  dil           10820 non-null  str           
 12  resim      

## 5. Yorumlar: yapı kontrolü

**Dosyalar:** `data/processed/yorumlar/` altında 23.107 parquet.
- oyunlar'a ait: **22.898** → SQLite'a yazılacak
- Yazılım: **209** → atlanacak
- Beklenen satır (`oyunlar.yorum_sayisi` toplamı): **31.690.738** (01'deki 31.832.390 − 141.652 yazılım yorumu)

**Tek dosya (Halls of Torment, 2218750):** 10.820 yorum × 15 sütun.

| Sütun | pandas tipi | SQLite'a |
|---|---|---|
| review_id | str (`"2218750_0"`) | TEXT, birincil anahtar |
| app_id | **str** | **INTEGER'a çevrilecek**, oyunlar'a yabancı anahtar |
| tarih | **datetime** | **`'YYYY-AA-GG'` metne çevrilecek** |
| oynama_saat, dil_guven | float | REAL |
| faydali_oy, uzunluk | int | INTEGER |
| metin, dil_ham, dil | str | TEXT (boş metin → NULL) |
| olumlu, erken_erisim, resim, kopya, chroma_uygun | bool | INTEGER 1/0 |

Boş metin: 60 yorum. `dil_ham` da onlarda boş, `dil` = "bilinmiyor". SQL'de kalır, Chroma'ya girmez.

In [6]:
# Yorumlar 2: 22.898 dosyayı SQLite'a yaz
import time
from tqdm.auto import tqdm

baglanti.execute("DROP TABLE IF EXISTS yorumlar")
baglanti.execute("""CREATE TABLE yorumlar (
    review_id    TEXT    PRIMARY KEY,
    app_id       INTEGER NOT NULL REFERENCES oyunlar(app_id),
    oynama_saat  REAL,
    tarih        TEXT,
    faydali_oy   INTEGER,
    metin        TEXT,
    olumlu       INTEGER,
    erken_erisim INTEGER,
    uzunluk      INTEGER,
    dil_ham      TEXT,
    dil_guven    REAL,
    dil          TEXT,
    resim        INTEGER,
    kopya        INTEGER,
    chroma_uygun INTEGER)""")

hatalar = []
baslangic = time.time()
for dosya in tqdm(yazilacak):
    try:
        yorum_sql = sqlite_icin_hazirla(pd.read_parquet(dosya))
        yorum_sql.to_sql("yorumlar", baglanti, if_exists="append", index=False)
    except Exception as e:
        hatalar.append((dosya.name, repr(e)))
baglanti.commit()
sure_dk = (time.time() - baslangic) / 60

yazilan = baglanti.execute("SELECT COUNT(*) FROM yorumlar").fetchone()[0]
print(f"Süre: {sure_dk:.1f} dk, hata: {len(hatalar)}")
print(f"Yazılan: {yazilan:,} / beklenen: {oyunlar['yorum_sayisi'].sum():,}")
print("foreign_key_check (ilk 10):", baglanti.execute("PRAGMA foreign_key_check(yorumlar)").fetchall()[:10])
print(f"steam.db boyutu: {DB_YOLU.stat().st_size / 1e9:.2f} GB")
for h in hatalar[:20]:
    print(h)

/home/kaan/steam-oyun-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 22898/22898 [12:52<00:00, 29.66it/s] 


Süre: 12.9 dk, hata: 0
Yazılan: 31,690,738 / beklenen: 31,690,738
foreign_key_check (ilk 10): []
steam.db boyutu: 13.38 GB


## 6. yorumlar tablosu → steam.db

22.898 parquet tek tek okundu, `sqlite_icin_hazirla` ile app_id int'e ve tarih `'YYYY-AA-GG'` metne çevrildi, `to_sql(if_exists="append")` ile yazıldı. 209 yazılım dosyası atlandı.

**Şema:** `review_id TEXT PRIMARY KEY`, `app_id INTEGER NOT NULL REFERENCES oyunlar(app_id)`, 15 sütun (metin dahil; bool'lar 1/0).

**Sonuç:**
| Ölçüm | Değer |
|---|---|
| Süre | 12,9 dk (~30 dosya/sn) |
| Hata | 0 |
| Yazılan / beklenen | 31.690.738 / 31.690.738 |
| `foreign_key_check(yorumlar)` | `[]` |
| steam.db boyutu | **13,38 GB** (ham CSV'ler 10,55 GB) |

**Boyut neden büyük:** 9 ek sütun (dil, bayraklar, uzunluk), review_id metin anahtarının ayrı indeksi (31,7 M kimlik iki kez saklanıyor) ve metin. Deploy aşamasında boyut tekrar ele alınacak.

**Durum:** 6 tablonun hepsi yazıldı. Sıradaki: indeksler.

In [7]:
# İndeksler: önce/sonra hız ölçümü
def olc(sorgu):
    plan = baglanti.execute("EXPLAIN QUERY PLAN " + sorgu).fetchall()
    baslangic = time.time()
    sonuc = baglanti.execute(sorgu).fetchall()
    return sonuc, time.time() - baslangic, [p[-1] for p in plan]

test_sorgulari = {
    "Halls of Torment olumsuz yorum": "SELECT COUNT(*) FROM yorumlar WHERE app_id = 2218750 AND olumlu = 0",
    "Türkçe yorum":                   "SELECT COUNT(*) FROM yorumlar WHERE dil = 'tr'",
    "2024 ve sonrası yorum":          "SELECT COUNT(*) FROM yorumlar WHERE tarih >= '2024-01-01'",
    "Horror etiketli oyun":           "SELECT COUNT(*) FROM oyun_etiket WHERE etiket = 'Horror'",
}

def hepsini_olc(baslik):
    print(f"--- {baslik} ---")
    for ad, sorgu in test_sorgulari.items():
        sonuc, sure, plan = olc(sorgu)
        print(f"{ad}: {sonuc[0][0]:,} | {sure:.2f} sn | plan: {plan}")
    print()

hepsini_olc("İndeks ÖNCESİ")

indeksler = [
    "CREATE INDEX IF NOT EXISTS idx_yorumlar_app_id ON yorumlar(app_id)",
    "CREATE INDEX IF NOT EXISTS idx_yorumlar_dil ON yorumlar(dil)",
    "CREATE INDEX IF NOT EXISTS idx_yorumlar_tarih ON yorumlar(tarih)",
    "CREATE INDEX IF NOT EXISTS idx_oyun_etiket_etiket ON oyun_etiket(etiket)",
    "CREATE INDEX IF NOT EXISTS idx_oyun_tur_tur ON oyun_tur(tur)",
    "CREATE INDEX IF NOT EXISTS idx_oyun_kategori_kategori ON oyun_kategori(kategori)",
    "CREATE INDEX IF NOT EXISTS idx_oyun_dil_dil ON oyun_dil(dil)",
]
for komut in indeksler:
    baslangic = time.time()
    baglanti.execute(komut)
    print(f"{komut.split()[5]}: {time.time() - baslangic:.1f} sn")
baglanti.execute("ANALYZE")
baglanti.commit()
print()

hepsini_olc("İndeks SONRASI")
print(f"steam.db boyutu: {DB_YOLU.stat().st_size / 1e9:.2f} GB (önce 13,38 GB)")

--- İndeks ÖNCESİ ---
Halls of Torment olumsuz yorum: 922 | 9.78 sn | plan: ['SCAN yorumlar']
Türkçe yorum: 619,054 | 8.43 sn | plan: ['SCAN yorumlar']
2024 ve sonrası yorum: 9,643,201 | 3.93 sn | plan: ['SCAN yorumlar']
Horror etiketli oyun: 5,751 | 0.09 sn | plan: ['SCAN oyun_etiket USING COVERING INDEX sqlite_autoindex_oyun_etiket_1']

idx_yorumlar_app_id: 9.7 sn
idx_yorumlar_dil: 25.7 sn
idx_yorumlar_tarih: 26.8 sn
idx_oyun_etiket_etiket: 0.5 sn
idx_oyun_tur_tur: 0.2 sn
idx_oyun_kategori_kategori: 0.2 sn
idx_oyun_dil_dil: 0.2 sn

--- İndeks SONRASI ---
Halls of Torment olumsuz yorum: 922 | 0.00 sn | plan: ['SEARCH yorumlar USING INDEX idx_yorumlar_app_id (app_id=?)']
Türkçe yorum: 619,054 | 0.01 sn | plan: ['SEARCH yorumlar USING COVERING INDEX idx_yorumlar_dil (dil=?)']
2024 ve sonrası yorum: 9,643,201 | 0.10 sn | plan: ['SEARCH yorumlar USING COVERING INDEX idx_yorumlar_tarih (tarih>?)']
Horror etiketli oyun: 5,751 | 0.00 sn | plan: ['SEARCH oyun_etiket USING COVERING INDEX idx_o

## 7. İndeksler

İndeks = sütundaki değerlerin sıralı bir kopyası + satır numaraları. SQLite tabloyu baştan sona okumak (`SCAN`) yerine indekste arayıp doğrudan satıra gider (`SEARCH ... USING INDEX`). Veri yazıldıktan sonra oluşturuldu (yazma sırasında olsa her satırda güncellenirdi).

**Oluşturulanlar:** yorumlar(app_id), yorumlar(dil), yorumlar(tarih), oyun_etiket(etiket), oyun_tur(tur), oyun_kategori(kategori), oyun_dil(dil). Sonra `ANALYZE` (planlayıcı için istatistik). oyunlar'a indeks yok (64.948 satır zaten milisaniyede taranıyor).

| Sorgu | Sonuç | Önce | Sonra | Plan (sonra) |
|---|---|---|---|---|
| Halls of Torment olumsuz yorum | 922 | 9,78 sn | 0,00 sn | SEARCH USING INDEX app_id |
| Türkçe yorum | 619.054 | 8,43 sn | 0,01 sn | COVERING INDEX dil |
| 2024 ve sonrası yorum | 9.643.201 | 3,93 sn | 0,10 sn | COVERING INDEX tarih |
| Horror etiketli oyun | 5.751 | 0,09 sn | 0,00 sn | COVERING INDEX etiket |

- **Covering index:** cevap tamamen indeksten çıkar, tabloya gidilmez.
- **Bedel:** oluşturma ~64 sn; steam.db 13,38 → **14,84 GB** (+1,46 GB).

In [8]:
# Kapanış: steam.db'nin son hâli
for tip, ad, tablo in baglanti.execute("""
        SELECT type, name, tbl_name FROM sqlite_master
        WHERE type IN ('table', 'index')
        ORDER BY tbl_name, type DESC, name"""):
    if tip == "table":
        satir = baglanti.execute(f"SELECT COUNT(*) FROM {ad}").fetchone()[0]
        print(f"\n{ad}: {satir:,} satır")
    else:
        print(f"   indeks: {ad}")

baglanti.close()
print(f"\nsteam.db: {DB_YOLU.stat().st_size / 1e9:.2f} GB, bağlantı kapatıldı")


oyun_dil: 341,659 satır
   indeks: idx_oyun_dil_dil
   indeks: sqlite_autoindex_oyun_dil_1

oyun_etiket: 730,709 satır
   indeks: idx_oyun_etiket_etiket
   indeks: sqlite_autoindex_oyun_etiket_1

oyun_kategori: 235,168 satır
   indeks: idx_oyun_kategori_kategori
   indeks: sqlite_autoindex_oyun_kategori_1

oyun_tur: 174,563 satır
   indeks: idx_oyun_tur_tur
   indeks: sqlite_autoindex_oyun_tur_1

oyunlar: 64,948 satır

sqlite_stat1: 13 satır

yorumlar: 31,690,738 satır
   indeks: idx_yorumlar_app_id
   indeks: idx_yorumlar_dil
   indeks: idx_yorumlar_tarih
   indeks: sqlite_autoindex_yorumlar_1

steam.db: 14.84 GB, bağlantı kapatıldı


## 8. Kapanış

`sqlite_master` ile steam.db'nin son hâli:

| Tablo | Satır | İndeksler |
|---|---|---|
| oyunlar | 64.948 | (app_id = rowid, ayrı indeks gerekmez) |
| oyun_tur | 174.563 | idx_oyun_tur_tur + anahtar |
| oyun_etiket | 730.709 | idx_oyun_etiket_etiket + anahtar |
| oyun_kategori | 235.168 | idx_oyun_kategori_kategori + anahtar |
| oyun_dil | 341.659 | idx_oyun_dil_dil + anahtar |
| yorumlar | 31.690.738 | idx_yorumlar_app_id, _dil, _tarih + anahtar (review_id) |

- `sqlite_autoindex_*`: birincil anahtar için SQLite'ın kendi oluşturduğu indeksler.
- `sqlite_stat1`: ANALYZE istatistikleri (13 satır).
- **steam.db: 14,84 GB.** Bağlantı kapatıldı; 04 kendi bağlantısını açacak.

**Sıradaki (04_sql_araci):** agent'ın SQL aracı: salt okuma, satır sınırı (kestiyse söyler), "oran her zaman oyunlar.olumlu_oran'dan" kuralı, tarih filtresini aralıkla yazma.